<a href="https://colab.research.google.com/github/thayusky/7DaysOfCode-ML/blob/main/Day2_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# #7DaysOfCode - Day 2: Data Preprocessing
Goal: clean and transform the Spotify data so it can be used to train a classification model that predicts whether a song will be popular or not.

## 1. Loading and cleaning the data

In [1]:
import kagglehub
import pandas as pd
import numpy as np

# Download and read the dataset
path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")
df = pd.read_csv(path + "/dataset.csv")

# Remove the extra index column and rows with missing values
df = df.drop(columns=["Unnamed: 0"])
df = df.dropna()

# Keep one row per track
tracks = df.drop_duplicates(subset="track_id").copy()

print(tracks.shape)

Using Colab cache for faster access to the '-spotify-tracks-dataset' dataset.
(89740, 20)


## 2. Removing misleading zero-popularity songs

On Day 1, we found that many zero-popularity songs are compilation or re-release versions of songs that have another, more popular version. Their popularity is zero because of how Spotify counts plays, not because of their musical features, so they could mislead the model. We will remove them.

In [2]:
# Mark songs that appear more than once with the same name and artists
tracks["has_other_version"] = tracks.duplicated(subset=["track_name", "artists"], keep=False)

# Songs to remove: zero popularity AND another version exists
to_remove = (tracks["popularity"] == 0) & (tracks["has_other_version"])

print("Songs removed:", to_remove.sum())

# Keep everything that is NOT in to_remove
tracks = tracks[~to_remove]

print(tracks.shape)

Songs removed: 5455
(84285, 21)


## 3. Creating the target variable

We will convert popularity into a binary class: songs with a popularity of 70 or above are considered popular (1), and the rest are considered not popular (0). This cut-off is a choice and can be adjusted to test different scenarios.

In [3]:
# Define the conditions and the value for each condition
conditions = [
    tracks["popularity"] >= 70,
    tracks["popularity"] < 70
]
choices = [1, 0]

# Create the new column using numpy.select()
tracks["popular"] = np.select(conditions, choices)

tracks[["track_name", "popularity", "popular"]].head(10)

,track_name,popularity,popular
0,Comedy,73,1
1,Ghost - Acoustic,55,0
2,To Begin Again,57,0
3,Can't Help Falling In Love,71,1
4,Hold On,82,1
5,Days I Will Remember,58,0
6,Say Something,74,1
7,I'm Yours,80,1
8,Lucky,74,1
9,Hunger,56,0


## 4. Checking class balance

In [4]:
# Count of each class
print(tracks["popular"].value_counts())

# Percentage of each class
print(tracks["popular"].value_counts(normalize=True) * 100)

popular
0    81159
1     3126
Name: count, dtype: int64
popular
0    96.291155
1     3.708845
Name: proportion, dtype: float64


**Finding:** the classes are highly imbalanced: only about 3.7% of songs are popular (3,126 out of 84,285). A model that predicts "not popular" for every song would be about 96.3% accurate without learning anything. Therefore, accuracy alone is misleading, and we will also use metrics such as precision, recall and AUC.

## 5. Selecting the features

The **features** (input variables, `X`) are the information the model uses to make a prediction. The **target** (output variable, `y`) is what we want to predict: `popular`.

We must **not** use as features:
- **`popularity` and `popular`**: they *are* the answer. Using them would be **data leakage**: the model would "cheat" and look brilliant in testing, but be useless in real life.
- **`has_other_version`**: a helper column we created for cleaning, not a property of the song.
- **Identifiers and free text** (`track_id`, `track_name`, `album_name`, `artists`): they identify a song but do not describe how it sounds.
- **`track_genre`**: after keeping one row per track, each song keeps only one of its genres (chosen arbitrarily), so we will leave it out for now.

We will use the musical features, plus `duration_ms` and `explicit`.

In [5]:
# Musical features that describe how a song sounds
features = [
    "duration_ms", "explicit", "danceability", "energy", "key",
    "loudness", "mode", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "time_signature"
]

# X = input variables (features), y = output variable (target)
X = tracks[features].copy()
y = tracks["popular"]

# Convert explicit from True/False to 1/0
X["explicit"] = X["explicit"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
X.dtypes

X shape: (84285, 14)
y shape: (84285,)


,0
duration_ms,int64
explicit,int64
danceability,float64
energy,float64
key,int64
loudness,float64
mode,int64
speechiness,float64
acousticness,float64
instrumentalness,float64


## 6. Normalisation: are the features on the same scale?

In [6]:
# Compare the minimum and maximum of each feature
X.describe().loc[["min", "max"]].T

,min,max
duration_ms,8586.000,5237295.000
explicit,0.000,1.000
danceability,0.000,0.985
energy,0.000,1.000
key,0.000,11.000
loudness,-49.531,4.532
mode,0.000,1.000
speechiness,0.000,0.965
acousticness,0.000,0.996
instrumentalness,0.000,1.000


**Finding:** the features are on very different scales. For example, `duration_ms` goes up to millions, `loudness` is negative, and most features range from 0 to 1. Some models (e.g. logistic regression, KNN) would give more weight to features with larger numbers, so we will need to **normalise** them.

**Important:** normalisation must be fitted on the **training data only**, after splitting the data into training and test sets. Otherwise, information from the test set would leak into training (another form of data leakage). We will do this when we split the data.

## 7. Day 2 summary

- Removed the extra index column, missing values and duplicated tracks.
- Removed 5,455 zero-popularity songs that were compilation or re-release versions of other songs.
- Created a binary target: popular (popularity ≥ 70) = 1, not popular = 0.
- Found a strong class imbalance: only about 3.7% of songs are popular.
- Selected 14 musical features and avoided data leakage.
- Identified the need for normalisation, to be applied after the train/test split.

## 8. Strategy for the class imbalance

Only about 3.7% of songs are popular, which is similar to modelling a rare species in ecology. Our strategy:

- **Metrics:** use precision, recall, balanced accuracy and AUC instead of accuracy alone (similar to sensitivity, specificity, TSS and AUC in species distribution models).
- **Class weights:** give more weight to the rare class (`class_weight="balanced"`), similar to prevalence weighting in species distribution models.
- **Stratified train/test split:** keep the same proportion of popular songs in the training and test sets.
- **Threshold:** the model returns a probability (like habitat suitability), and we will choose a threshold that balances sensitivity and specificity, rather than always using 0.5.